In [4]:
from PIL import Image

def extract_data(stego_image_path):
    img = Image.open(stego_image_path)
    img = img.convert('RGB')
    pixels = img.load()
    width, height = img.size
    max_capacity = width * height * 3
    
    extracted_bits = ""
    target_data_len = None # ตัวแปรเก็บว่าเราต้องเดินกี่ก้าว
    
    print("🔍 กำลังสแกนหาข้อมูลที่ซ่อนอยู่ในภาพ...")

    for y in range(height):
        for x in range(width):
            r, g, b = pixels[x, y]
            
            extracted_bits += str(r % 2)
            extracted_bits += str(g % 2)
            extracted_bits += str(b % 2)
            
            # 1. ถ้ายังไม่รู้ความยาว ให้ดึง 32 บิตแรกมาอ่านก่อน
            if target_data_len is None and len(extracted_bits) >= 32:
                header_bits = extracted_bits[:32]
                target_data_len = int(header_bits, 2)
                
                # เช็คความปลอดภัย ถ้าความยาวเกินขนาดรูป แปลว่ารูปนี้ไม่ได้ซ่อนอะไรไว้
                if target_data_len == 0 or target_data_len > max_capacity:
                    print("❌ ไม่พบข้อมูลลับ (หรือภาพไม่ได้ถูกซ่อนข้อมูลไว้)")
                    return ""
                    
            # 2. ถ้ารู้ความยาวแล้ว ก็ดึงข้อมูลไปเรื่อยๆ จนกว่าจะครบตามจำนวน
            if target_data_len is not None and len(extracted_bits) >= 32 + target_data_len:
                # เอาเฉพาะข้อมูลเนื้อๆ (ตัด 32 บิตแรกที่เป็นตัวบอกความยาวทิ้ง)
                secret_data = extracted_bits[32 : 32 + target_data_len]
                print("✅ ดึงข้อมูลครบถ้วน 100% ไม่มีขาดไม่มีเกิน!")
                return secret_data

    return ""

if __name__ == "__main__":
    stego_image = r"D:\Users\ASUS\Desktop\mini-root\output_cat.png" 
    output_secret_img = r"D:\Users\ASUS\Desktop\mini-root\secret_extracted.png"
    
    print("=== โปรเจกต์ Shadow-Pixel: ระบบถอดรหัส ===")
    
    try:
        extracted_binary = extract_data(stego_image)
        
        if extracted_binary:
            print("⚙️ กำลังประกอบร่าง 0101 กลับเป็นไฟล์...")
            
            byte_array = bytearray()
            for i in range(0, len(extracted_binary), 8):
                chunk = extracted_binary[i:i+8]
                if len(chunk) == 8:
                    byte_array.append(int(chunk, 2))
                    
            if byte_array.startswith(b'TXT|'):
                final_message = byte_array[4:].decode('utf-8', errors='ignore')
                print(f"\n🎉 ความลับคือข้อความ: '{final_message}'")
                
            elif byte_array.startswith(b'IMG|'):
                img_bytes = byte_array[4:]
                with open(output_secret_img, "wb") as f:
                    f.write(img_bytes)
                print(f"\n🎉 ความลับคือรูปภาพ! บันทึกเป็นไฟล์ 'secret_extracted.png' เรียบร้อย สมบูรณ์แน่นอน!")
                
            else:
                print("❌ ข้อมูลที่ดึงได้ไม่มีป้ายแท็ก (การดึงข้อมูลอาจจะผิดพลาด)")
                
    except FileNotFoundError:
        print(f"❌ หาไฟล์ output_cat.png ไม่เจอ!")

=== โปรเจกต์ Shadow-Pixel: ระบบถอดรหัส ===
🔍 กำลังสแกนหาข้อมูลที่ซ่อนอยู่ในภาพ...
✅ ดึงข้อมูลครบถ้วน 100% ไม่มีขาดไม่มีเกิน!
⚙️ กำลังประกอบร่าง 0101 กลับเป็นไฟล์...

🎉 ความลับคือรูปภาพ! บันทึกเป็นไฟล์ 'secret_extracted.png' เรียบร้อย สมบูรณ์แน่นอน!
